# select correct samples from gsm8k

In [ ]:
# 给前1319行样例文件id添加test, 后面所有的id添加train
import json
# jsonl_file = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/out/llama2-7b-chat-hf/cot0shot/eval_all_all/0/gsm8k_bottom_0.000000_GSM8K_cot0shot.jsonl"
jsonl_file = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/out/llama2-7b-chat-hf/direct/eval_all_all/0/gsm8k_bottom_0.000000_GSM8K_cot0shot.jsonl"

modified_lines = []
with open(jsonl_file, "r") as file:
    for i, line in enumerate(file):
        entry = json.loads(line.strip())
        if i < 1319:
            entry["id"] = "test_" + entry["id"]
        else:
            entry["id"] = "train_" + entry["id"]
        modified_lines.append(entry)

# Write the modified data back to the file
with open(jsonl_file, "w") as file:
    for entry in modified_lines:
        file.write(json.dumps(entry) + "\n")

In [ ]:
import json
import random

jsonl_file_direct = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/out/llama2-7b-chat-hf/direct/eval_all_all/0/gsm8k_bottom_0.000000_GSM8K_cot0shot.jsonl"
jsonl_file_cot0shot = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/out/llama2-7b-chat-hf/cot0shot/eval_all_all/0/gsm8k_bottom_0.000000_GSM8K_cot0shot.jsonl"
jsonl_file_ids = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K_eval_build/eval_dataset_with_conversation_template.jsonl"
correct_ids = []
direct_by_id, cot0shot_by_id = {}, {}

# read direct
with open(jsonl_file_direct, "r") as f_direct:
    for raw in f_direct:
        obj = json.loads(raw.strip())
        direct_by_id[obj["id"]] = obj
        if obj.get("correct"):
            correct_ids.append(obj["id"])

# read cot0shot
with open(jsonl_file_cot0shot, "r") as f_cot:
    for raw in f_cot:
        obj = json.loads(raw.strip())
        cot0shot_by_id[obj["id"]] = obj
        if obj.get("correct"):
            correct_ids.append(obj["id"])

print(len(correct_ids))

id_eval = 

# Build helper sets
direct_correct_ids_set = {qid for qid, rec in direct_by_id.items() if rec.get("correct")}
cot0shot_correct_ids_set = {qid for qid, rec in cot0shot_by_id.items() if rec.get("correct")}

# 1) Calibration datasets (120 random correct samples from each file)
random.seed(42)
calibration_direct_ids = random.sample(list(direct_correct_ids_set), min(120, len(direct_correct_ids_set)))
calibration_cot0shot_ids = random.sample(list(cot0shot_correct_ids_set), min(120, len(cot0shot_correct_ids_set)))

calibration_direct = [direct_by_id[qid] for qid in calibration_direct_ids]
calibration_cot0shot = [cot0shot_by_id[qid] for qid in calibration_cot0shot_ids]

print(f"Calibration (direct): {len(calibration_direct)}")
print(f"Calibration (cot0shot): {len(calibration_cot0shot)}")

# 2) Eval dataset components
direct_correct_cot_correct_samples = [
    qid for qid in (direct_correct_ids_set & cot0shot_correct_ids_set)
]
cot0shot_correct_direct_wrong_samples = [
    qid for qid in cot0shot_correct_ids_set - direct_correct_ids_set
]

calib_union_ids = set(calibration_direct_ids) | set(calibration_cot0shot_ids)

both_correct_eval_pool = [qid for qid in direct_correct_cot_correct_samples if qid not in calib_union_ids]
cot_correct_direct_wrong_eval_pool = [qid for qid in cot0shot_correct_direct_wrong_samples if qid not in calib_union_ids]

random.seed(42)
selected_direct_correct_cot_correct = random.sample(both_correct_eval_pool, min(300, len(both_correct_eval_pool)))
selected_cot0shot_correct_direct_wrong = random.sample(cot_correct_direct_wrong_eval_pool, min(300, len(cot_correct_direct_wrong_eval_pool)))

# Final eval IDs
eval_ids = selected_direct_correct_cot_correct + selected_cot0shot_correct_direct_wrong

# --- 写出 direct.jsonl 和 cot0shot.jsonl ---
eval_direct = []
eval_cot0shot = []
for qid in eval_ids:
    if qid in direct_by_id:
        rec_direct = dict(direct_by_id[qid])
        rec_direct["sample_type"] = "both_correct" if qid in selected_direct_correct_cot_correct else "cot0shot_correct_direct_wrong"
        eval_direct.append(rec_direct)
    if qid in cot0shot_by_id:
        rec_cot = dict(cot0shot_by_id[qid])
        rec_cot["sample_type"] = "both_correct" if qid in selected_direct_correct_cot_correct else "cot0shot_correct_direct_wrong"
        eval_cot0shot.append(rec_cot)

with open("/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K_eval_build/eval_direct.jsonl", "w") as f_out:
    for rec in eval_direct:
        f_out.write(json.dumps(rec) + "\n")

with open("/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K_eval_build/eval_cot0shot.jsonl", "w") as f_out:
    for rec in eval_cot0shot:
        f_out.write(json.dumps(rec) + "\n")

print(f"Eval saved: direct {len(eval_direct)}, cot0shot {len(eval_cot0shot)}")


3027
Calibration (direct): 120
Calibration (cot0shot): 120
Eval saved: direct 452, cot0shot 452


In [12]:
import json, random, os

# 路径
jsonl_file_cot4shot = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/out/llama2-7b-chat-hf/cot4shot/eval_all_all/0/gsm8k_bottom_0.000000_GSM8K_cot0shot.jsonl"
jsonl_file_cot0shot_goldreason = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/out/llama2-7b-chat-hf/cot0shot_goldreason/eval_all_all/0/gsm8k_bottom_0.000000_GSM8K_cot0shot.jsonl"

# load
def load_jsonl(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            obj = json.loads(line.strip())
            data.append(obj)
    return {obj["id"]: obj for obj in data}

cot4shot_by_id = load_jsonl(jsonl_file_cot4shot)
cot0shot_goldreason_by_id = load_jsonl(jsonl_file_cot0shot_goldreason)

# 正确样本 id 集合（取两个文件的交集，以保证 ID 在两边都有）
cot4shot_correct_ids = {qid for qid, rec in cot4shot_by_id.items() if rec.get("correct")}
cot0shot_goldreason_correct_ids = {qid for qid, rec in cot0shot_goldreason_by_id.items() if rec.get("correct")}
valid_ids = cot4shot_correct_ids & cot0shot_goldreason_correct_ids

print(f"cot4shot correct: {len(cot4shot_correct_ids)}")
print(f"cot0shot_goldreason correct: {len(cot0shot_goldreason_correct_ids)}")
print(f"Intersection (valid for calibration): {len(valid_ids)}")

# 随机选择120个 ID
random.seed(42)
calibration_cot0shot_ids = random.sample(list(valid_ids), min(120, len(valid_ids)))

# 在两个文件中取相同 ID
calibration_from_cot4shot = [cot4shot_by_id[qid] for qid in calibration_cot0shot_ids]
calibration_from_cot0shot_goldreason = [cot0shot_goldreason_by_id[qid] for qid in calibration_cot0shot_ids]

# 保存
out_dir = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K_eval_build"
os.makedirs(out_dir, exist_ok=True)

def save_jsonl(path, rows):
    with open(path, "w") as f:
        for rec in rows:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl(os.path.join(out_dir, "calibration_cot4shot_120.jsonl"), calibration_from_cot4shot)
save_jsonl(os.path.join(out_dir, "calibration_cot0shot_goldreason_120.jsonl"), calibration_from_cot0shot_goldreason)

print("✅ Saved:")
print(f"- calibration_cot4shot_120.jsonl ({len(calibration_from_cot4shot)})")
print(f"- calibration_cot0shot_goldreason_120.jsonl ({len(calibration_from_cot0shot_goldreason)})")
print(f"Shared calibration IDs: {len(calibration_cot0shot_ids)}")


cot4shot correct: 708
cot0shot_goldreason correct: 7471
Intersection (valid for calibration): 644
✅ Saved:
- calibration_cot4shot_120.jsonl (120)
- calibration_cot0shot_goldreason_120.jsonl (120)
Shared calibration IDs: 120


In [10]:
import os

out_dir = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K_eval_build"
os.makedirs(out_dir, exist_ok=True)

# 保存函数
def save_jsonl(path, rows):
    with open(path, "w") as f:
        for rec in rows:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# 保存 calibration
calib_direct_path = os.path.join(out_dir, "calibration_direct_120_with_conversation_template.jsonl")
calib_cot_path = os.path.join(out_dir, "calibration_cot0shot_120_with_conversation_template.jsonl")
save_jsonl(calib_direct_path, calibration_direct)
save_jsonl(calib_cot_path, calibration_cot0shot)

# 保存 eval dataset
eval_path = os.path.join(out_dir, "eval_dataset.jsonl")
save_jsonl(eval_path, eval_dataset)

print("✅ Files saved:")
print(f"- Calibration direct: {calib_direct_path} ({len(calibration_direct)})")
print(f"- Calibration cot0shot: {calib_cot_path} ({len(calibration_cot0shot)})")
print(f"- Eval dataset: {eval_path} ({len(eval_dataset)})")


✅ Files saved:
- Calibration direct: /common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K_eval_build/calibration_direct_120_with_conversation_template.jsonl (120)
- Calibration cot0shot: /common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K_eval_build/calibration_cot0shot_120_with_conversation_template.jsonl (120)
- Eval dataset: /common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K_eval_build/eval_dataset.jsonl (454)


In [ ]:
# 将两个文件中重复的问题去掉，生成一个新的文件

import json

file1 = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K/combined_selected_samples_600.jsonl"
file2 = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K/output/calibration_set_cot_120.json"
output_file = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K/combined_selected_samples_600_no_overlap_with_calibration.jsonl"

# 1. 读取calibration_set_cot_120.json中的所有question
with open(file2, "r") as f:
    calibration_data = json.load(f)
calibration_questions = set()
if isinstance(calibration_data, list):
    for entry in calibration_data:
        q = entry.get("question")
        if q is not None:
            calibration_questions.add(q)
elif isinstance(calibration_data, dict):
    for entry in calibration_data.values():
        q = entry.get("question")
        if q is not None:
            calibration_questions.add(q)

print(f"Calibration set questions: {len(calibration_questions)}")

# 2. 读取combined_selected_samples_600.jsonl，去除与calibration_questions重复的question
filtered_entries = []
removed_count = 0
with open(file1, "r") as f:
    for line in f:
        if line.strip():
            entry = json.loads(line)
            q = entry.get("question")
            if q is not None and q not in calibration_questions:
                filtered_entries.append(entry)
            else:
                removed_count += 1

print(f"Original combined_selected_samples_600: {len(filtered_entries) + removed_count}")
print(f"Removed due to overlap: {removed_count}")
print(f"Remaining after deduplication: {len(filtered_entries)}")

# 3. 保存去重后的新文件
with open(output_file, "w") as f:
    for entry in filtered_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"去重后的文件已保存到: {output_file}")

Calibration set questions: 120
Original combined_selected_samples_600: 600
Removed due to overlap: 14
Remaining after deduplication: 586
去重后的文件已保存到: /common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K/combined_selected_samples_600_no_overlap_with_calibration.jsonl


In [2]:
# 检查correct_ids中有没有重复的
# Check for duplicates in correct_ids
unique_ids = set(correct_ids)

print(f"Total correct_ids: {len(correct_ids)}")
print(f"Unique correct_ids: {len(unique_ids)}")
print(f"Duplicates found: {len(correct_ids) - len(unique_ids)}")

if len(correct_ids) != len(unique_ids):
    # Find and display duplicates
    from collections import Counter
    id_counts = Counter(correct_ids)
    duplicates = {id_val: count for id_val, count in id_counts.items() if count > 1}
    print(f"Duplicate IDs: {duplicates}")
else:
    print("No duplicates found in correct_ids")


NameError: name 'correct_ids' is not defined

In [ ]:
# 检查这个文件中有没有重复的ids
import json


# Load JSONL file (one JSON object per line)
data = []
with open("/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K/combined_selected_samples_600.jsonl", "r") as f:
    for line in f:
        if line.strip():  # Skip empty lines
            data.append(json.loads(line))

# 检查这个文件中有没有重复的ids
ids = [entry["id"] for entry in data]
unique_ids = set(ids)

print(f"Total ids: {len(ids)}")
print(f"Unique ids: {len(unique_ids)}")
print(f"Duplicates found: {len(ids) - len(unique_ids)}")

if len(ids) != len(unique_ids):
    # Find and display duplicates
    from collections import Counter
    id_counts = Counter(ids)

In [ ]:
len(cot4shot_correct_direct_wrong_samples)

In [ ]:
import json
import pandas as pd
import random
random.seed(42)
# Load the two JSON files
file_direct = "../data/GSM8K/output/output.GSM8K.direct.math_teacher.llama2-7b-chat.json"
file_cot = "../data/GSM8K/output/output.GSM8K.cot0shot.math_teacher.llama2-7b-chat.json"

with open(file_direct, "r") as f:
    data_direct = json.load(f)

with open(file_cot, "r") as f:
    data_cot = json.load(f)

# Convert to dictionaries indexed by question ID for easy comparison
dict_direct = {entry["id"]: entry for entry in data_direct}
dict_cot = {entry["id"]: entry for entry in data_cot}

# 1. Find the calibration set: CoT result is True and Direct result is False
calibration_set = []
for qid in dict_direct:
    if qid in dict_cot:
        if dict_cot[qid].get("cot0shot.math teacher_result") and not dict_direct[qid].get("direct.math teacher_result"):
            calibration_set.append(qid)

# 2. Question IDs of the calibration set
calibration_ids = calibration_set

# 3. choose these calibration_ids samples from these two files and save them
calibration_data_cot = [dict_cot[qid] for qid in calibration_ids]   
# with open("../data/GSM8K/output/calibration_set_cot.json", "w") as f:
    # json.dump(calibration_data_cot, f, indent=4)

direct_true_ids = [qid for qid, entry in dict_direct.items() if entry.get("direct.math teacher_result") is True]
direct_true_samples = [dict_direct[qid] for qid in direct_true_ids]

# 4. Calibration set of direct should be all direct.math teacher_resul true samples plus some other samples
# find the true samples in direct
num_extra = 120 - len(direct_true_ids)  # number of extra samples needed
calibration_data_direct = [dict_direct[qid] for qid in calibration_ids]
calibration_candidates = [entry for entry in calibration_data_direct if entry["id"] not in direct_true_ids]
extra_samples = random.sample(calibration_candidates, min(num_extra, len(calibration_candidates)))

direct_calibration_samples = direct_true_samples + extra_samples

# with open("../data/GSM8K/output/calibration_set_direct_mixed.json", "w") as f:
#     json.dump(direct_calibration_samples, f, indent=4)


In [ ]:
len(cot4shot_correct_direct_wrong_samples)

In [ ]:
import json, random, re, os
from pathlib import Path

# ---------- ① 读入两份结果 ---------- #
file_direct = "../data/GSM8K/output/output.GSM8K.direct.math_teacher.llama2-7b-chat.json"
file_cot    = "../data/GSM8K/output/output.GSM8K.cot0shot.math_teacher.llama2-7b-chat.json"

with open(file_direct) as f:
    data_direct = json.load(f)
with open(file_cot) as f:
    data_cot = json.load(f)

dict_direct = {e["id"]: e for e in data_direct}
dict_cot    = {e["id"]: e for e in data_cot}

# ---------- ② 校准集 ID（两份文件共有部分） ---------- #
all_calibration_set = [qid for qid in dict_direct if qid in dict_cot]

# 提取其中的数字部分 → {15, 42, …}
exclude_ids_digit = {
    int(qid.replace("GSM8K_Q", ""))
    for qid in all_calibration_set
}


# ---------- ③ 构造 held-out ID 列表 ---------- #
TOTAL = 1319                       # GSM8K 总样本数
K     = 500                        # 目标 held-out 数量
random.seed(42)                    # 可复现

all_digits      = set(range(1, TOTAL + 1))
candidate_ids   = list(all_digits - exclude_ids_digit)
heldout_digits  = random.sample(candidate_ids, K)
# ---------- ④ 构造 held-out 样本 ---------- #
heldout_records = []
for digit in heldout_digits:
    qid = f"GSM8K_Q{digit}"
    record = {
        "id": qid,
    }
    heldout_records.append(record)

# ---------- ⑤ 写入 JSONL ---------- #
data_file = Path(
    "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K/heldout_500.jsonl"
)
data_file.parent.mkdir(parents=True, exist_ok=True)

with open(data_file, "w") as f:
    for rec in heldout_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"已保存到 {data_file}")


In [ ]:
len(heldout_digits)

In [ ]:
len(direct_true_samples)

In [ ]:
len(calibration_ids)

In [ ]:

# # 3. randomly choose 100 samples not in the calibration set as the evaluation set
# /common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K/test.jsonl
evaluation_ids = [qid for qid in dict_direct if qid not in calibration_ids][:100]



In [ ]:
# 1. find the difference sets of these two files as the calibration set
"cot0shot.math teacher_result": true while "direct.math teacher_result": false
# 2. the question id of the calibration sets
# 3. find 100 sample set except the calibration set as the evaluation set

In [ ]:
alignment-attribution-code/data/GSM8K/output/output.GSM8K.direct.math_teacher.llama2-7b-chat.json 
alignment-attribution-code/data/GSM8K/output/output.GSM8K.cot0shot.math_teacher.llama2-7b-chat.json